# Deri Lezyonu Segmentasyonu — Klasik Görüntü İşleme

Bu notebook, dermoskopi görüntülerinde lezyon sınırını **derin öğrenme kullanmadan**
bulmayı ve dört farklı yöntemi adil biçimde karşılaştırmayı amaçlıyor.

**Soru:** Klasik görüntü işleme, dermoskopi artefaktları (kıl, vinyet, düşük kontrast)
temizlendiğinde ne kadar başarılı olabilir?

**Yaklaşım**

| Aşama | Yapılan |
|-------|---------|
| Ön işleme | DullRazor kıl temizleme → median filtre → kuadratik gölge düzeltme → CLAHE |
| Segmentasyon | Otsu · Adaptif Gaussian · K-means (LAB) · Watershed |
| Son işleme | Açma → küçük bölge eleme → delik doldurma → kapama → bölge seçimi |
| Değerlendirme | IoU, Dice, duyarlılık, kesinlik, XOR hatası, HD95, ASSD + eşleştirilmiş test |

Kodun tamamı `src/` altındaki modüllerde; notebook yalnızca onları çağırıp sonucu yorumluyor.

In [ ]:
import sys
from pathlib import Path

# Notebook notebooks/ içinden çalıştırıldığında proje kökünü yola ekle
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import LesionDataset, DatasetNotFound
from src.pipeline import Pipeline, build_pipelines, ablation_pipelines
from src.preprocess import preprocess, PreprocessConfig, hair_mask
from src.metrics import compute_metrics, align, METRIC_NAMES, LOWER_IS_BETTER
from src.synthetic import synthetic_batch
from src import viz

plt.rcParams["figure.dpi"] = 110
print("Hazir.")

## 1. Veri

Gerçek veri seti bulunursa o kullanılır; bulunamazsa notebook sentetik örneklerle çalışır.
Böylece kod her ortamda çalışır, ama **sentetik sonuçlar gerçek performans değildir** —
bu ayrımı aşağıdaki `USING_REAL_DATA` bayrağıyla takip ediyoruz.

In [ ]:
N_EVAL = 150          # toplu değerlendirmede kullanılacak görüntü sayısı
DATA_ROOT = None      # ör. "data" veya "/kaggle/input/.../data"

try:
    dataset = LesionDataset(DATA_ROOT, limit=N_EVAL)
    USING_REAL_DATA = len(dataset) > 0
except DatasetNotFound as error:
    print(f"Veri seti bulunamadi -> sentetik moda geciliyor.\n  ({error})")
    USING_REAL_DATA = False

if USING_REAL_DATA:
    print(dataset)
    samples = [sample.load() for sample in dataset]
else:
    samples = list(synthetic_batch(count=min(N_EVAL, 40), seed=0))
    print(f"{len(samples)} sentetik ornek uretildi.")

image, truth = samples[0]
print("Goruntu boyutu:", image.shape, "| lezyon piksel orani:", round(float(truth.mean()), 4))

## 2. Ön işleme

Dermoskopi görüntülerinde segmentasyonu bozan üç artefakt var ve her birinin ayrı bir çaresi:

1. **Kıl/tüy** — lezyonun üstünü çizgi halinde keser, eşiklemede yanlış kenar üretir.
   *DullRazor*: dört yönde doğrusal yapısal elemanla morfolojik kapama yapılır, kapanmış
   görüntüyle orijinal arasındaki fark kılları verir, o bölgeler inpaint ile doldurulur.
2. **Vinyet** — köşelerdeki karartma. Lezyon da koyu olduğu için eşikleme köşeleri lezyon sanar.
   Parlaklık kanalına **yalnızca deri piksellerinden** kuadratik yüzey oturtup bölüyoruz.
   (Geniş Gaussian bulanıklaştırma ile aydınlatma tahmini yapmak büyük lezyonlarda
   lezyonun kendisini "aydınlatma" sayıp kontrastı yok ediyordu.)
3. **Düşük yerel kontrast** — LAB uzayında yalnızca L kanalına CLAHE; a/b kanallarına
   dokunulmadığı için lezyonun rengi bozulmaz.

In [ ]:
stages = preprocess(image)
fig = viz.show_stages(stages, title="Ön işleme zinciri")
plt.show()

print("Kil olarak isaretlenen piksel orani:", round(float(stages["hair_mask"].mean()), 4))
print("Gecerli bolge (FOV) orani:", round(float(stages["fov"].mean()), 4))

### Ön işleme gerçekten gerekli mi?

Adımları tek tek kapatıp Dice'taki düşüşe bakıyoruz (ablasyon). Bir adım kapatıldığında
skor düşmüyorsa o adım bu veri için gereksiz demektir.

In [ ]:
def evaluate_pipelines(pipelines, samples, with_boundary=True):
    """Her örnek için her pipeline'ı çalıştırıp uzun formatlı DataFrame döndürür."""
    rows = []
    for index, (img, gt) in enumerate(samples):
        for name, pipeline in pipelines.items():
            metrics = pipeline.evaluate(img, gt, with_boundary=with_boundary)
            rows.append({"image": index, "method": name, **metrics})
    return pd.DataFrame(rows)


ablation = evaluate_pipelines(ablation_pipelines("otsu"), samples[:30], with_boundary=False)
ablation_summary = (
    ablation.groupby("method")[["dice", "iou", "sensitivity", "precision"]]
    .mean()
    .sort_values("dice", ascending=False)
)
ablation_summary["dice_kaybi"] = ablation_summary.loc["tam", "dice"] - ablation_summary["dice"]
ablation_summary.round(4)

## 3. Segmentasyon yöntemleri

| Yöntem | Fikir | Güçlü yanı | Zayıf yanı |
|--------|-------|-----------|-----------|
| **Otsu** | Histogramı iki sınıfa ayıran tek global eşik | Hızlı, parametresiz | Aydınlatma düzgün değilse veya lezyon/deri kontrastı düşükse bozulur |
| **Adaptif** | Eşik her piksel için komşuluğundan | Düzgün olmayan aydınlatmaya dayanıklı | Geniş düz lezyonun içini kaçırır (iç bölgede yerel kontrast yok) — delik doldurma şart |
| **K-means** | LAB uzayında renk kümeleme, en koyu küme | Parlaklık farkı az ama renk farkı belirginse iyi | Başlangıca duyarlı, daha yavaş |
| **Watershed** | Otsu'dan işaretleyici, gradyana göre havza | Sınırı gerçek kenara oturtur, bitişik yapıları ayırır | Yanlış işaretleyici → aşırı/eksik bölütleme |

Dördü de aynı ön işleme ve aynı son işlemeyi kullanıyor; aradaki tek fark eşik kararı.
Karşılaştırmanın adil olması için bu önemli.

In [ ]:
pipelines = build_pipelines()
fig = viz.compare_methods(image, truth, pipelines)
plt.show()

## 4. Metrikler

Yalnızca örtüşmeye bakmak yetersiz:

- **IoU / Dice** — bölge örtüşmesi. Standart, ama sınır kalitesine duyarsız.
- **Accuracy** — yanıltıcı olabilir: lezyon görüntünün %10'uysa, her şeye "deri" diyen
  bir model %90 doğruluk alır. Bu yüzden tek başına raporlanmamalı.
- **XOR hatası** — `(FP+FN) / lezyon alanı`. Küçük lezyonlarda accuracy'den çok daha ayırt edici.
- **HD95 / ASSD** — sınır mesafeleri (piksel). Aynı Dice'a sahip iki maske çok farklı
  sınır kalitesinde olabilir; bunlar onu yakalar. **Küçük değer iyidir.**

In [ ]:
single = {}
for name, pipeline in pipelines.items():
    single[name] = pipeline.evaluate(image, truth)

single_df = pd.DataFrame(single).T[list(METRIC_NAMES)]
single_df.round(4)

## 5. Toplu değerlendirme

Tek görüntüden sonuç çıkarılmaz. Aşağıda tüm örnekler üzerinde ortalama, standart sapma
ve dağılım hesaplanıyor.

In [ ]:
results = evaluate_pipelines(pipelines, samples)

summary = results.groupby("method")[list(METRIC_NAMES)].agg(["mean", "std"])
overview = pd.DataFrame({
    "dice_ort": summary[("dice", "mean")],
    "dice_std": summary[("dice", "std")],
    "iou_ort": summary[("iou", "mean")],
    "duyarlilik": summary[("sensitivity", "mean")],
    "kesinlik": summary[("precision", "mean")],
    "xor_hata": summary[("xor_error", "mean")],
    "hd95": summary[("hd95", "mean")],
    "assd": summary[("assd", "mean")],
}).sort_values("dice_ort", ascending=False)

print(f"n = {results['image'].nunique()} goruntu | gercek veri: {USING_REAL_DATA}")
overview.round(4)

In [ ]:
means = {
    method: results.loc[results["method"] == method, list(METRIC_NAMES)].mean().to_dict()
    for method in results["method"].unique()
}
fig = viz.plot_metric_bars(means)
plt.show()

fig = viz.plot_distributions(results, metric="dice")
plt.show()

### Fark anlamlı mı?

Ortalamalar birbirine yakın çıktığında "A, B'den iyidir" demek için yeterli kanıt yoktur.
Aynı görüntüler üzerinde **eşleştirilmiş Wilcoxon işaretli sıra testi** uyguluyoruz;
bu test normallik varsayımı gerektirmez (Dice dağılımı genelde sola çarpıktır).

In [ ]:
from scipy.stats import wilcoxon

pivot = results.pivot_table(index="image", columns="method", values="dice").dropna()
methods = list(pivot.columns)

rows = []
for i, a in enumerate(methods):
    for b in methods[i + 1:]:
        diff = pivot[a].to_numpy() - pivot[b].to_numpy()
        stat, p = (np.nan, 1.0) if np.allclose(diff, 0) else wilcoxon(pivot[a], pivot[b])
        rows.append({
            "A": a, "B": b,
            "dice_A": pivot[a].mean(), "dice_B": pivot[b].mean(),
            "fark": diff.mean(),
            "A_kazandi_%": 100 * (diff > 0).mean(),
            "p": p,
            "anlamli": p < 0.05,
        })

pd.DataFrame(rows).round(4)

## 6. Nerede başarısız oluyor?

Ortalama skor, yöntemin **neden** hata yaptığını söylemez. En kötü örneklere bakmak
sınırları anlamanın en hızlı yolu.

In [ ]:
worst = results.groupby("image")["dice"].mean().sort_values().index[:3]

for index in worst:
    img_i, gt_i = samples[index]
    score = results.loc[results["image"] == index, "dice"].mean()
    fig = viz.compare_methods(img_i, gt_i, pipelines)
    fig.suptitle(f"Zor ornek #{index} — ortalama Dice = {score:.3f}", fontsize=11)
    plt.show()

## 7. Sonuç

> Aşağıdaki yorumları **kendi çalıştırdığın sayılara göre** doldur. Sentetik modda çıkan
> yüksek skorlar gerçek veri performansı değildir.

Beklenen genel örüntü ve nedenleri:

- **Ön işleme katkısı.** Ablasyon tablosunda en büyük düşüşü hangi adımın kapatılması
  yaratıyorsa, bu veri setinin baskın artefaktı odur. Kıllı görüntülerde DullRazor,
  köşeleri kararmış görüntülerde gölge düzeltmesi öne çıkar.
- **Otsu vs adaptif.** Otsu, lezyon ve deri histogramda iki belirgin tepe oluşturduğunda
  çok iyidir; geçişin yumuşak olduğu durumlarda eşiği kaydırır. Adaptif yöntem iç bölgeyi
  kaçırdığı için delik doldurma adımına bağımlıdır.
- **Accuracy'ye güvenme.** Lezyon oranı düşük görüntülerde accuracy ~0.95 çıkarken
  Dice 0.4 olabilir. XOR hatası ve HD95 bu durumu açığa çıkarır.
- **Sınır kalitesi.** Watershed genelde Dice'ta devrim yaratmaz ama HD95/ASSD'de
  iyileşme sağlar — sınırı gerçek kenara oturtur.

**Bu yaklaşımın sınırları:** çoklu lezyon, lezyonun kadrajı taşması, mürekkep işaretleri,
balon/jel yansımaları ve açık renkli (amelanotik) lezyonlar klasik eşiklemenin zorlandığı
durumlar. Bunlar için öğrenme tabanlı yöntemler gerekir.